In [ ]:
#driver para entrar
! pip install mysql-connector-python

In [ ]:
#conexion directorio base del drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime
import time
import logging
import json
from dotenv import load_dotenv
import uuid

In [ ]:
import os

# Permite que el usuario defina la ruta, o usa un valor por defecto
ruta_base = os.getenv("RUTA_DATOS", "./datos/")

ruta_credencial = f"{ruta_base}credenciales/"
ruta_log = f"{ruta_base}log/"
ruta_destino = f"{ruta_base}add/parquet"
archivo_log = f"{ruta_log}logs_adquisicion.log"

# Crear carpetas si no existe
os.makedirs(ruta_credencial, exist_ok=True)
os.makedirs(ruta_log, exist_ok=True)
os.makedirs(ruta_destino, exist_ok=True)

In [ ]:
# configuracion de los logs
logger = logging.getLogger("pipeLine-logger")
logger.setLevel(logging.INFO)

# si hay algo que este manejando, elimino el manejador del log
if logger.hasHandlers():
    logger.handlers.clear()

file_handler = logging.FileHandler(archivo_log)
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)
print("logger configurado")

In [ ]:
# identificador unico del script
run_id = str(uuid.uuid4())
inicio_proceso = time.time()
estado_global = "exitoso"

# inicializar variables
registros_extraidos = 0
registros_cargados = 0

In [ ]:
run_id

In [ ]:
logger.info(f"Inicio de pipline | Run ID: {run_id}")
print(f"Inicio de pipline | Run ID: {run_id}")

In [ ]:
# variables de entorno para credenciales
env_path = f"{ruta_credencial}.env"
load_dotenv(env_path)

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

print(DB_USER)

In [ ]:
connection_string = f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
print(connection_string)
engine = create_engine(connection_string)

In [ ]:
try:
  with engine.connect() as conn:
    print("conexion exitosa")
except Exception as e:
  estado_global = "error"
  logger.error(f"Error al conectar a la base de datos: {e}")
  print(f"Error al conectar la base de datos: {e}")

In [ ]:
#Adquisición incremental
LAST_PROCESSED_DATE = "2022-12-31"

query = """
SELECT clave_alu, nombre, ap_paterno, ap_materno, sexo, curp, fedita
FROM alumnos
WHERE YEAR(fedita) = YEAR(%s)
"""

In [ ]:
try:
  df = pd.read_sql(query, engine, params=(LAST_PROCESSED_DATE,))
  print("Registros leidos", len(df))
  registros_extraidos = len(df)
  #Registro de auditoría
  log_info = {
        "run_id": run_id,
        "fecha_proceso": datetime.now().isoformat(),
        "evento": "extraccion",
        "registros": len(df),
        "fuente": "mysql_alumnos",
        "estado": "ok"
  }
  logger.info(f"Proceso de adquisición: {json.dumps(log_info)}")
  print("Log escrito")

except Exception as e:
  estado_global = "error"
  logger.error(f"Error al extraer los datos: {e}")
  print(f"Error al extraer los datos: {e}")

In [ ]:
df

In [ ]:
#Control de volumen y escalabilidad
count_query = "SELECT YEAR(fedita) as anio, COUNT(*) as total FROM alumnos GROUP BY YEAR(fedita) ORDER BY 1;"
df_count = pd.read_sql(count_query, engine)
print(df_count)

In [ ]:
df.info()
df.isnull().sum()

In [ ]:
df["nombre_completo"] = df["ap_paterno"].str.upper() + " " + df["ap_materno"].str.upper() + " " + df["nombre"].str.upper()
df.drop(columns=["nombre", "ap_paterno", "ap_materno"], inplace=True)
df

In [ ]:
fecha_incremental = LAST_PROCESSED_DATE.replace("-", "")
try:
  nombre_archivo = f"alumnos_{fecha_incremental}.parquet"
  df.to_parquet(f"{ruta_destino}{nombre_archivo}", index=False)
  registros_cargados += len(df)
  #Registro de auditoría
  log_info = {
        "run_id": run_id,
        "fecha_proceso": datetime.now().isoformat(),
        "evento": "extraccion",
        "registros": len(df),
        "fuente": "mysql_alumnos",
        "estado": "ok"
  }
  logger.info(f"Proceso de adquisición: {json.dumps(log_info)}")
  print("Log escrito")

except Exception as e:
  estado_global = "error"
  logger.error(f"Error al extraer los datos: {e}")
  print(f"Error al extraer los datos: {e}")

In [ ]:
print(ruta_log)
! cat {archivo_log}

In [ ]:
! ls -la {ruta_base}

In [ ]:
! cat {ruta_destino}{nombre_archivo}

In [ ]:
print(ruta_destino)
! ls -la {ruta_destino}

# Segunda Parte

In [ ]:
autor = "AUTOR"

# Destino (analítica)
engine_destino = create_engine(
    f"mysql+mysqlconnector://"
    f"{os.getenv('DST_DB_USER')}:"
    f"{os.getenv('DST_DB_PASSWORD')}@"
    f"{os.getenv('DST_DB_HOST')}:"
    f"{os.getenv('DST_DB_PORT')}/"
    f"{os.getenv('DST_DB_NAME')}"
)

In [ ]:
#SQL parametrizado
query = """
SELECT %s AS autor,
       a.clave_alu,
       CONCAT_WS(' ', a.ap_paterno, a.ap_materno, nombre) AS alumno,
       SUM(p.pago) AS tpago,
       COUNT(p.pago) AS npago
FROM alumnos a
LEFT JOIN pagos p ON a.clave_alu = p.clave_alu
WHERE YEAR(fecha_pago) = %s
GROUP BY a.clave_alu
ORDER BY a.clave_alu
"""
df = pd.read_sql(query, engine, params=(autor,2018, ))
registros_extraidos += len(df)
df

In [ ]:
#Renombrar columnas
df.rename(columns={'clave_alu':'matricula'}, inplace=True)
#Organizar las columnas conforme a la tabla destino
#Filtrar solo alumnos con promedio
df["promedio_calificacion"] = 0
df["n_evaluaciones"] = 0
df["fuente"] = "mysql"

#Select para organizar el orden de mis columnas
df = df[
    [
        "autor",
        "matricula",
        "alumno",
        "tpago",
        "npago",
        "promedio_calificacion",
        "n_evaluaciones",
        "fuente"
    ]
]
df

In [ ]:
#tabla a donde se va a llegar
tabla_destino = "alumnos_metricas_dw"

try:
  logger.info(f"Iniciando carga a tabla de destino {tabla_destino}")
  with engine_destino.connect() as conn:
    df.to_sql(
        tabla_destino,
        conn,
        if_exists="append",
        index=False
    )

    log_info = {
        "evento": "carga_destino",
        "tabla": tabla_destino,
        "registros_insertados": len(df),
        "estado": "exitoso",
        "timestamp": datetime.now().isoformat()
    }

    registros_cargados += len(df)
    logger.info(f"Carga finalizada {json.dumps(log_info)}")
    print("Log escrito")

except Exception as e:
  estado_global = "error"
  logger.error(f"Error al cargar los datos: {e}")
  print(f"Error al cargar datos: {e}")

In [ ]:
# cerrar el proceso, hora y fecha de temino
fin_proceso = time.time()
duracion_total = round(fin_proceso - inicio_proceso, 2)

log_info = {
    "run_id": run_id,
    "evento": "fin_pipeline",
    "estado_global": estado_global,
    "registros_extraidos": registros_extraidos,
    "registros_cargados": registros_cargados,
    "duracion_segundos": duracion_total
}
logger.info(f"Fin de pipeline: {json.dumps(log_info)}")

#Cerrar el log
logging.shutdown()

print("Pipeline finalizado")
print("Run ID:", run_id)
print("Duración:", duracion_total, "segundos")

In [ ]:
print(ruta_log)
! cat {archivo_log}